# 12 — Optimized Batch Pipeline

This notebook addresses the delays observed in the previous run.

The normal fresh-topic path now has approximately four local-model calls:

1. outline;
2. script;
3. claim extraction;
4. combined verification and corrected script.

The separate editor call is skipped. A retry can be faster because saved
outlines/scripts, claims, web evidence, and completed fact-check decisions are
reused.

The notebook does not embed the final video in its output.

## Load the project

In [1]:
import sqlite3
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.batch import TopicBatchPlan
from educational_shorts.batch_optimized import (
    OptimizedBatchConfig,
    preflight_batch_pipeline,
    run_batch_pipeline_optimized,
    summarize_batch_manifest,
)
from educational_shorts.schemas import VideoTopicList
from educational_shorts.topic_library import (
    claim_next_topic,
    mark_topic_completed,
    mark_topic_failed,
    reset_failed_topic,
)

print("NOTEBOOK_12_BATCH_PIPELINE_OPTIMIZED")
print(f"Project root: {PROJECT_ROOT}")

NOTEBOOK_12_BATCH_PIPELINE_OPTIMIZED
Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
ROOT_CATEGORY = "Science"
TOPIC_CATEGORY_ROOT = ROOT_CATEGORY
TOPIC_DATABASE_PATH = PROJECT_ROOT / "data" / "topic_library.db"

# Enter a failed database topic ID to retry that exact topic.
# Leave as None to claim the next approved topic normally.
RETRY_FAILED_TOPIC_ID = None

PAUSE_ON_MANUAL_REVIEW = False
CONTINUE_ON_ERROR = False
SKIP_EXISTING_FINAL = True

# Major speed settings.
SKIP_SCRIPT_EDITOR = True
REUSE_EXISTING_INTERMEDIATES = True
FACT_CHECK_MAX_CLAIMS = 4
FACT_CHECK_MAX_SEARCH_RESULTS = 5
FACT_CHECK_MAX_SOURCES_PER_CLAIM = 2
FACT_CHECK_MAX_EXCERPT_CHARS = 700
FACT_CHECK_RETRIEVAL_WORKERS = 4
FACT_CHECK_MINIMUM_WORDS = 85
FACT_CHECK_MAXIMUM_WORDS = 150
USE_CLAIM_CACHE = True
USE_FACT_CHECK_DECISION_CACHE = True
FORCE_REFRESH_FACT_CHECK = False

GAMEPLAY_SUBDIRECTORY = "subway_surfers"
BACKGROUND_VIDEO_FILENAME = None
TTS_VOICE = "am_michael"
TTS_SPEED = 1.0

CAPTION_STYLE = "phrase"
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
CROP_ANCHOR_Y = "top"
LOOP_BACKGROUND = True

config = OptimizedBatchConfig(
    root_category=ROOT_CATEGORY,
    pause_on_manual_review=PAUSE_ON_MANUAL_REVIEW,
    continue_on_error=CONTINUE_ON_ERROR,
    skip_existing_final=SKIP_EXISTING_FINAL,
    skip_script_editor=SKIP_SCRIPT_EDITOR,
    reuse_existing_intermediates=(
        REUSE_EXISTING_INTERMEDIATES
    ),
    fact_check_max_claims=FACT_CHECK_MAX_CLAIMS,
    fact_check_max_search_results=(
        FACT_CHECK_MAX_SEARCH_RESULTS
    ),
    fact_check_max_sources_per_claim=(
        FACT_CHECK_MAX_SOURCES_PER_CLAIM
    ),
    fact_check_max_excerpt_chars=(
        FACT_CHECK_MAX_EXCERPT_CHARS
    ),
    fact_check_retrieval_workers=(
        FACT_CHECK_RETRIEVAL_WORKERS
    ),
    fact_check_minimum_words=FACT_CHECK_MINIMUM_WORDS,
    fact_check_maximum_words=FACT_CHECK_MAXIMUM_WORDS,
    use_claim_cache=USE_CLAIM_CACHE,
    use_fact_check_decision_cache=(
        USE_FACT_CHECK_DECISION_CACHE
    ),
    force_refresh_fact_check=FORCE_REFRESH_FACT_CHECK,
    gameplay_subdirectory=GAMEPLAY_SUBDIRECTORY,
    background_video_filename=BACKGROUND_VIDEO_FILENAME,
    tts_voice=TTS_VOICE,
    tts_speed=TTS_SPEED,
    caption_style=CAPTION_STYLE,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
    output_width=OUTPUT_WIDTH,
    output_height=OUTPUT_HEIGHT,
    crop_anchor_y=CROP_ANCHOR_Y,
    loop_background=LOOP_BACKGROUND,
)

print(config.model_dump_json(indent=2))

{
  "root_category": "Science",
  "tree_filename": null,
  "category_path_override": null,
  "topic_candidate_count": 10,
  "min_category_depth": 2,
  "max_category_depth": 3,
  "category_selection_seed": 42,
  "topic_generation_seed": 42,
  "topic_generation_temperature": 0.7,
  "target_seconds": 60,
  "section_count": 4,
  "outline_temperature": 0.4,
  "outline_seed": 42,
  "target_words_per_minute": 145,
  "script_temperature": 0.5,
  "script_seed": 42,
  "editor_minimum_seconds": 40,
  "editor_maximum_seconds": 60,
  "editor_temperature": 0.4,
  "editor_seed": 42,
  "fact_check_minimum_words": 85,
  "fact_check_maximum_words": 150,
  "fact_check_temperature": 0.1,
  "fact_check_seed": 42,
  "metadata_temperature": 0.5,
  "metadata_seed": 42,
  "metadata_max_attempts": 4,
  "pause_on_manual_review": false,
  "tts_language_code": "a",
  "tts_voice": "am_michael",
  "tts_speed": 1.0,
  "tts_chunk_pause_ms": 80,
  "tts_segment_pause_ms": 240,
  "tts_target_peak_dbfs": -1.0,
  "normaliz

## Preflight check

In [3]:
preflight = preflight_batch_pipeline(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in preflight.items():
    print(f"{name}: {value}")

project_root: c:\Users\hitch\python_files\educational_shorts
tree_path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
tree_root: Science
gameplay_directory: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers
background_video: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers\ScreenRecording_07-23-2026 10-14-28_1.mp4
ffmpeg: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffprobe.EXE


## Claim a topic

When `RETRY_FAILED_TOPIC_ID` is set, this cell resets that row and claims the
same topic directly. This avoids accidentally taking a different approved
topic.

In [4]:
def claim_topic_by_id(
    database_path: Path,
    topic_id: int,
):
    reset_failed_topic(
        database_path=database_path,
        topic_id=topic_id,
    )

    with sqlite3.connect(database_path) as connection:
        connection.row_factory = sqlite3.Row
        connection.execute("BEGIN IMMEDIATE")

        row = connection.execute(
            '''
            SELECT id, title, learning_objective, category_path
            FROM topics
            WHERE id = ? AND status = 'approved'
            ''',
            (topic_id,),
        ).fetchone()

        if row is None:
            connection.rollback()
            raise RuntimeError(
                f"Topic ID {topic_id} is not available as approved."
            )

        selected_at = datetime.now(timezone.utc).isoformat()
        connection.execute(
            '''
            UPDATE topics
            SET status = 'processing',
                selected_at_utc = ?,
                failure_message = NULL
            WHERE id = ?
            ''',
            (selected_at, topic_id),
        )
        connection.commit()

    import json
    from educational_shorts.schemas import VideoTopic

    category_path = json.loads(row["category_path"])
    topic = VideoTopic(
        title=row["title"],
        learning_objective=row["learning_objective"],
        category_path=category_path,
    )
    return int(row["id"]), topic, category_path


if RETRY_FAILED_TOPIC_ID is None:
    claimed_topic = claim_next_topic(
        database_path=TOPIC_DATABASE_PATH,
        category_root=TOPIC_CATEGORY_ROOT,
    )
else:
    claimed_topic = claim_topic_by_id(
        database_path=TOPIC_DATABASE_PATH,
        topic_id=RETRY_FAILED_TOPIC_ID,
    )

if claimed_topic is None:
    raise RuntimeError(
        "No approved topics are available in the topic library."
    )

topic_id, selected_topic, category_path = claimed_topic
now = datetime.now(timezone.utc)

topic_plan = TopicBatchPlan(
    run_id=now.strftime("%Y%m%dT%H%M%S%fZ"),
    category_path=category_path,
    topics_file=str(TOPIC_DATABASE_PATH),
    created_at_utc=now.isoformat(),
    topics=VideoTopicList(topics=[selected_topic]),
)

print(f"Database topic ID: {topic_id}")
print(f"Topic: {selected_topic.title}")
print(f"Category: {' > '.join(category_path)}")
print(f"Learning objective: {selected_topic.learning_objective}")
print("Database status: processing")

Database topic ID: 2
Topic: How Do Bacteria Communicate?
Category: Science > Biology > Microbiology
Learning objective: Understand how bacteria use chemical signals to communicate and coordinate behavior.
Database status: processing


## Run the optimized pipeline

Progress appears immediately for every stage. On a retry, existing files print
`[REUSE]` instead of running the earlier model calls again.

In [5]:
started = time.perf_counter()

try:
    batch_manifest = run_batch_pipeline_optimized(
        project_root=PROJECT_ROOT,
        plan=topic_plan,
        selected_topic_indexes=[0],
        config=config,
    )

    item = batch_manifest.items[0]

    if item.status in {"completed", "skipped_existing"}:
        final_video = item.paths.get("final_video")

        if not final_video:
            raise RuntimeError(
                "The pipeline reported completion but did not provide "
                "a final video path."
            )

        mark_topic_completed(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
            final_video_path=Path(final_video),
        )
        print("Topic-library status: completed")

    elif item.status == "manual_review":
        reset_failed_topic(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
        )
        print("Topic-library status: requeued for manual review")

    else:
        mark_topic_failed(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
            message=f"{item.status}: {item.message}",
        )
        print("Topic-library status: failed")

except Exception as error:
    mark_topic_failed(
        database_path=TOPIC_DATABASE_PATH,
        topic_id=topic_id,
        message=f"{type(error).__name__}: {error}",
    )
    raise

finally:
    elapsed_minutes = (time.perf_counter() - started) / 60
    print(f"Total notebook run time: {elapsed_minutes:.1f} minutes")

Optimized pipeline enabled: editor skipped=True, max claims=4, verification and rewrite combined into one model call.

ITEM 1/1: How Do Bacteria Communicate?
[START] Outline generation
[REUSE] Outline: c:\Users\hitch\python_files\educational_shorts\data\outlines\how_do_bacteria_communicate.json
[DONE] Outline generation: 0.0 min
[START] Script generation
[REUSE] Script: c:\Users\hitch\python_files\educational_shorts\data\scripts\how_do_bacteria_communicate.json
[DONE] Script generation: 0.0 min
[START] Script editing
[REUSE] Edited script: c:\Users\hitch\python_files\educational_shorts\data\edited_scripts\how_do_bacteria_communicate.json
[DONE] Script editing: 0.0 min
[START] Claim extraction
[DONE] Claim extraction: 3.0 min
Claims selected for checking: 4
[START] Evidence retrieval
  Retrieved 1/4: 2 source(s) from cache
  Retrieved 2/4: 2 source(s) from cache
  Retrieved 3/4: 2 source(s) from cache
  Retrieved 4/4: 2 source(s) from web
[DONE] Evidence retrieval: 0.1 min
[START] Combi

c:\Users\hitch\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\hitch\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Synthesizing 1/6: hook
Synthesizing 2/6: What Is Quorum Sensing?
Synthesizing 3/6: How Do They Send Messages?
Synthesizing 4/6: What Happens When They Communicate?
Synthesizing 5/6: Why It Matters
Synthesizing 6/6: closing
[DONE] Kokoro TTS: 1.0 min
[START] Caption generation
[DONE] Caption generation: 0.0 min
[START] Video assembly
[DONE] Video assembly: 3.2 min

Stage timing:
  Outline generation: 0.0 min
  Script generation: 0.0 min
  Script editing: 0.0 min
  Claim extraction: 3.0 min
  Evidence retrieval: 0.1 min
  Combined verification and rewrite: 12.4 min
  Metadata generation: 2.7 min
  Kokoro TTS: 1.0 min
  Caption generation: 0.0 min
  Video assembly: 3.2 min
  measured total: 22.4 min
Status: completed
Final stage: video_assembly
Finished the optimized educational-short pipeline.
Topic-library status: completed
Total notebook run time: 22.4 minutes


## Review the result

In [6]:
for name, value in summarize_batch_manifest(
    batch_manifest
).items():
    print(f"{name}: {value}")

item = batch_manifest.items[0]

print()
print(f"Status: {item.status}")
print(f"Final stage: {item.final_stage}")
print(f"Message: {item.message}")
print(f"Fact-check verdict: {item.fact_check_verdict}")
print(f"Requires manual review: {item.requires_manual_review}")

print()
print("Completed stages:")
for stage in item.stages_completed:
    print(f"  - {stage}")

print()
print("Saved paths:")
for name, path in item.paths.items():
    print(f"  {name}: {path}")

run_id: 20260724T173559696464Z
status: completed
selected_topics: 1
completed: 1
manual_review: 0
skipped_existing: 0
failed: 0
output_directory: c:\Users\hitch\python_files\educational_shorts\data\batch_runs\20260724T173559696464Z
manifest: c:\Users\hitch\python_files\educational_shorts\data\batch_runs\20260724T173559696464Z\batch_manifest.json

Status: completed
Final stage: video_assembly
Message: Finished the optimized educational-short pipeline.
Fact-check verdict: manual_review
Requires manual review: True

Completed stages:
  - outline_generation
  - script_generation
  - script_editing_reused
  - claim_extraction
  - evidence_retrieval
  - fact_checking
  - metadata_generation
  - tts_generation
  - caption_generation
  - video_assembly

Saved paths:
  outline: c:\Users\hitch\python_files\educational_shorts\data\outlines\how_do_bacteria_communicate.json
  script: c:\Users\hitch\python_files\educational_shorts\data\scripts\how_do_bacteria_communicate.json
  edited_script: c:\Use

## Final video path

The video is not embedded in the notebook, preventing a large notebook file.

In [7]:
final_video_path = item.paths.get("final_video")

if final_video_path:
    print(final_video_path)
else:
    print("No final video was produced.")

c:\Users\hitch\python_files\educational_shorts\data\videos\bacteria_talk_with_chemical_signals\final.mp4
